#### 교차검증과 그리드 서치
- 머신러닝을 사용할때 모델의 정확도를 측정하기 위해 반드시 사용해야 하는 방법    
- 딥러닝에서는 데이터의 크기가 크므로 이 방법은 필요 없다.  

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [3]:
# Feature와 Target
data = wine.iloc[:,:3].to_numpy()
target = wine.iloc[:,3].to_numpy()

In [6]:
wine.shape

(6497, 4)

----
#### 검증세트(Valid Set)추기
- 전체 과정에서 훈련세트와 테스트세트만 가지고 작업을 하면 테스트 세트 작업이 파라미터 값을 조절하여 정확성을 높힌다면 실전에는 아무런 의미가 없다.  
- 이런 과정을 방지하기 위해 훈련세트, 검증세트, 테스트세트로 구분하여 작업을 한다.  

In [7]:
# 전체 세트중 훈련세트와 테스트세트를 8:2의 기준으로 분리한다. 
from sklearn.model_selection import train_test_split

In [8]:
train_input, test_input, train_target, test_target = \
                        train_test_split(
                                data,
                                target,
                                test_size=0.2,
                                random_state=42
                        )

In [9]:
# 훈련세트중 훈련세트와 검증세트를 8:2의 기준으로 분리 
sub_input, val_input, sub_target, val_target = \
                        train_test_split(
                                train_input,
                                train_target,
                                test_size=0.2,
                                random_state=42
                        )

In [10]:
# 훈련세트, 검증세트, 테스트 세트의 크기
print("Train :", sub_input.shape)
print("Valid :", val_input.shape)
print("Test  :", test_input.shape)

Train : (4157, 3)
Valid : (1040, 3)
Test  : (1300, 3)


#### 결정트리

In [11]:
from sklearn.tree import DecisionTreeClassifier

In [12]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print("Train :", dt.score(sub_input, sub_target))
print("Valid :", dt.score(val_input, val_target))

Train : 0.9971133028626413
Valid : 0.864423076923077


---
#### 교차검증
- 교차검증은 한 파트를 폴드라고 하며 교차 검증의 기본 Fold는 5이다. 
- 훈련세트와 검증세트를 바꾸어 가며 정확도를 구하는 방법이다.  
- 전체의 정확도는 해당 값들의 평균으로 구한다. 

In [13]:
from sklearn.model_selection import cross_validate

In [14]:
scores = cross_validate(
                dt, 
                train_input,
                train_target,
                cv=5
)
scores

{'fit_time': array([0.00695586, 0.00700259, 0.00764561, 0.00700068, 0.00658774]),
 'score_time': array([0.0009985 , 0.        , 0.00100017, 0.0010016 , 0.        ]),
 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}

In [ ]:
# 교차검증후의 정확도 판단 
scores['test_score'].mean()

0.855300214703487

---
#### kFold를 이용한 방법 : Target의 분할 비율을 조정

In [17]:
from sklearn.model_selection import StratifiedKFold

In [18]:
splitter = StratifiedKFold(n_splits=5) # default : 5

scores = cross_validate(
                dt, 
                train_input,
                train_target,
                cv=splitter
)
scores

{'fit_time': array([0.00723338, 0.00687838, 0.00782657, 0.0059998 , 0.00700164]),
 'score_time': array([0.0010004 , 0.00099945, 0.        , 0.00100064, 0.00063848]),
 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}

In [19]:
scores['test_score'].mean()

0.855300214703487

In [20]:
# kFold를 10개로 나누어 교차검증

splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) # default : 5

scores = cross_validate(
                dt, 
                train_input,
                train_target,
                cv=splitter
)
scores

{'fit_time': array([0.00824857, 0.00592256, 0.01351857, 0.00900221, 0.00800157,
        0.00845098, 0.00031757, 0.01730299, 0.00973344, 0.        ]),
 'score_time': array([1.00040436e-03, 0.00000000e+00, 1.00016594e-03, 1.00421906e-03,
        1.00040436e-03, 1.00016594e-03, 0.00000000e+00, 1.00111961e-03,
        6.58035278e-05, 0.00000000e+00]),
 'test_score': array([0.83461538, 0.87884615, 0.85384615, 0.85384615, 0.84615385,
        0.87307692, 0.85961538, 0.85549133, 0.85163776, 0.86705202])}

In [21]:
scores['test_score'].mean()

0.8574181117533719

----
#### 그리드서치(GridSearch)를 이용한 최적의 Hyper Parameter 찾기 

##### 결정트리의 Hyper Parameter값 찾기

In [22]:
from sklearn.model_selection import GridSearchCV

In [23]:
# 일정한 값을 정한다. 
params = {'min_impurity_decrease' : [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [24]:
gs = GridSearchCV(
        DecisionTreeClassifier(
            random_state=42
        ),
        params,
        n_jobs=-1
)

In [25]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [26]:
dt2 = gs.best_estimator_
print(dt2.score(train_input, train_target))
print(dt2.score(test_input, test_target))

0.9615162593804117
0.8653846153846154


In [27]:
# 최적의 파라미터 값
gs.best_params_

{'min_impurity_decrease': 0.0001}

---
### Random Search(랜덤 서치)
: 정해진 파라미터가 아니고 파라미터의 범위를 정해 최적의 값 찾기

In [28]:
from scipy.stats import uniform, randint # 균등 분포 

In [31]:
# EX : uniform
ugen = uniform(0, 1)
ugen.rvs(5)

array([0.33089761, 0.63419576, 0.1745864 , 0.28430614, 0.80770094])

In [32]:
params = {
    'min_impurity_decrease' : uniform(0.0001, 0.001),
    'max_depth' : randint(20, 50),
    'min_samples_split' : randint(2, 25),
    'min_samples_leaf' : randint(1, 25)
}

In [33]:
from sklearn.model_selection import RandomizedSearchCV

In [34]:
gs = RandomizedSearchCV(
        DecisionTreeClassifier(random_state=42),
        params,
        n_iter=100,
        n_jobs=-1,
        random_state=42
)

In [35]:
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000023F678CDCA0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x0000023F68BEC500>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000023F68B37AD0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x0000023F68B34800>},
                   random_state=42)

In [36]:
# 최적의 파라미터 값
gs.best_params_

{'max_depth': 39,
 'min_impurity_decrease': 0.00034102546602601173,
 'min_samples_leaf': 7,
 'min_samples_split': 13}

In [37]:
gs.cv_results_

{'mean_fit_time': array([0.00610514, 0.00560112, 0.00620112, 0.00540109, 0.00560136,
        0.00480113, 0.00477419, 0.00520129, 0.00500121, 0.00520105,
        0.0048008 , 0.0044939 , 0.00440135, 0.00662198, 0.00558071,
        0.00500121, 0.00440111, 0.00520115, 0.00500097, 0.00580144,
        0.00540123, 0.00500121, 0.00520115, 0.00500093, 0.00440111,
        0.00560131, 0.00460114, 0.00480084, 0.00480113, 0.00424237,
        0.00440068, 0.00460129, 0.00440097, 0.00520139, 0.00520096,
        0.0048007 , 0.00500102, 0.00500112, 0.00540118, 0.00460072,
        0.00480123, 0.00461082, 0.00441046, 0.00720148, 0.00456901,
        0.00886216, 0.00500097, 0.00647802, 0.0059289 , 0.004601  ,
        0.00526161, 0.00540104, 0.00440073, 0.00546145, 0.00460076,
        0.00498466, 0.004633  , 0.00520101, 0.0048614 , 0.00552878,
        0.00600119, 0.00467353, 0.0043787 , 0.00600123, 0.00660152,
        0.00479069, 0.00460095, 0.00480084, 0.00460105, 0.0047895 ,
        0.00760164, 0.00538974,

In [38]:
gs.cv_results_['mean_test_score'].mean()


0.8639124620567113

In [39]:
dt3 = gs.best_estimator_
dt3.score(test_input, test_target)

0.86